In [ ]:
code = 'SRE_SI'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SRE(bt, start_time, end_time, orderside, sl, intra_sl, om, std_indicator, indicator_buffer, buffer_minute):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        
        # checking straddle indicator
        std_indicator_flag, std_indicator_time = bt.straddle_indicator(start_dt, end_dt, std_indicator, indicator_buffer, buffer_minute)
        
        if std_indicator_flag:
            start_dt = std_indicator_time
        else:
            return None

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, _, std_sl_time, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, with_ohlc=True)
        
        first_straddle = [f"({ce_scrip}, {pe_scrip})", std_open, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl]
        
        re_straddles = []
        for re_no in range(max_re):
            
            if std_sl_time and re_no < re_entries and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):
                start_dt = std_sl_time
                ce_scrip, pe_scrip, ce_price, pe_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om)
                
                if ce_scrip is None:
                    std_sl_time = ''
                    re_straddles.extend(['', '', False, False, '', 0])
                    continue

                std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, _, std_sl_time, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, with_ohlc=True)
                re_straddles.extend([f"({ce_scrip}, {pe_scrip})", std_open, std_sl_flag, std_intra_sl_flag, std_sl_time, std_pnl])
            else:
                re_straddles.extend(['', '', False, False, '', 0])

        return [code, bt.index, start_time, end_time, orderside, sl, intra_sl, om, std_indicator, indicator_buffer, buffer_minute, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + first_straddle + re_straddles
    
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, sl, intra_sl, om, std_indicator, indicator_buffer, buffer_minute])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re, re_entries = 7, 7

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_SL/P_intraSL/P_OM/P_StdIndicator/P_IndicatorBuffer/P_BufferMinute/Date/Day/DTE/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/STD{r}.Scrip/STD{r}.Open/STD{r}.SL.Flag/STD{r}.IntraSL/STD{r}.SL.Time/STD{r}.PNL'
            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SRE(bt, row.entry_time, row.exit_time, row.orderside, row.sl, row.intra_sl, row.om, row.std_indicator, row.indicator_buffer, row.buffer_minute) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))